In [1]:
import pandas as pd
import numpy as np
from statsmodels.formula.api import ols

from funciones import *

In [2]:
# Abrir archivo raw_data
data_folder = "../data"
df = pd.read_parquet(f"{data_folder}/clean_data.parquet")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype   
---  ------                          --------------  -----   
 0   Beta                            503 non-null    float64 
 1   DividendYield                   503 non-null    float64 
 2   ReturnOnAssets                  503 non-null    float64 
 3   profitMargins                   503 non-null    float64 
 4   operatingMargins                503 non-null    float64 
 5   currentRatio                    503 non-null    float64 
 6   revenueGrowth                   503 non-null    float64 
 7   shortPercentOfFloat             503 non-null    float64 
 8   PriceToBook_Transformed         503 non-null    float64 
 9   returnOnEquity_Transformed      503 non-null    float64 
 10  ForwardPE_Transformed           503 non-null    float64 
 11  MarketCap_log                   503 non-null    float64 
 12  debtToEquity_log      

In [3]:
# Generar aleatorios
df['random1'] = np.random.uniform(0,1,size=df.shape[0])
df['random2'] = np.random.uniform(0,1,size=df.shape[0])

# Se separan las variables objetivo y los inputs de predictores
varObjCont = df['MarketCap_log']

imput = df.drop(['MarketCap_log', 'Ticker'],axis=1)

# Ranking V de Cramer vs. obj continua
tablaCramer = pd.DataFrame(imput.apply(lambda x: cramers_v(x,varObjCont)),columns=['VCramer'])

px.bar(tablaCramer,x=tablaCramer.VCramer,title='Relaciones frente al precio').update_yaxes(categoryorder="total ascending")

In [19]:
# Matriz de correlaciones
corr = imput.select_dtypes(include=np.number).corr()
corr.style.background_gradient(cmap='viridis').format(precision=3)

,Beta,DividendYield,ReturnOnAssets,profitMargins,operatingMargins,currentRatio,revenueGrowth,shortPercentOfFloat,PriceToBook_Transformed,returnOnEquity_Transformed,ForwardPE_Transformed,debtToEquity_log,trailingPegRatio_log,EnterpriseToEbitda_Transformed,random1,random2
Beta,1.000,-0.364,0.128,0.074,0.038,0.284,0.200,0.110,0.185,0.101,0.068,-0.099,-0.268,0.305,-0.055,0.023
DividendYield,-0.364,1.000,-0.247,-0.075,0.031,-0.263,-0.200,0.092,-0.243,-0.200,-0.158,0.250,0.148,-0.304,-0.004,-0.045
ReturnOnAssets,0.128,-0.247,1.000,0.434,0.410,0.218,0.105,-0.063,0.220,0.623,0.084,-0.117,0.016,0.179,0.062,0.062
profitMargins,0.074,-0.075,0.434,1.000,0.759,0.079,0.142,-0.277,0.122,0.459,0.164,-0.163,0.129,0.162,0.086,-0.033
operatingMargins,0.038,0.031,0.410,0.759,1.000,0.052,0.198,-0.225,0.057,0.306,0.013,-0.099,0.058,0.136,0.078,-0.012
currentRatio,0.284,-0.263,0.218,0.079,0.052,1.000,0.085,0.129,0.136,-0.020,0.033,-0.458,-0.117,0.210,0.056,0.032
revenueGrowth,0.200,-0.200,0.105,0.142,0.198,0.085,1.000,-0.135,0.193,0.065,0.151,-0.081,-0.151,0.321,-0.043,-0.011
shortPercentOfFloat,0.110,0.092,-0.063,-0.277,-0.225,0.129,-0.135,1.000,-0.091,-0.134,-0.159,0.042,-0.118,-0.156,0.033,-0.011
PriceToBook_Transformed,0.185,-0.243,0.220,0.122,0.057,0.136,0.193,-0.091,1.000,0.480,0.255,0.168,0.073,0.273,-0.051,-0.020
returnOnEquity_Transformed,0.101,-0.200,0.623,0.459,0.306,-0.020,0.065,-0.134,0.480,1.000,0.069,0.297,-0.020,0.040,-0.014,-0.012


In [20]:
# Formula Primer modelo OLS
form = ols_formula(df, "MarketCap_log", 'Ticker')

In [21]:
form

'MarketCap_log ~ Beta + DividendYield + ReturnOnAssets + profitMargins + operatingMargins + currentRatio + revenueGrowth + shortPercentOfFloat + PriceToBook_Transformed + returnOnEquity_Transformed + ForwardPE_Transformed + debtToEquity_log + trailingPegRatio_log + EnterpriseToEbitda_Transformed + Sector + random1 + random2'

In [22]:
modelo1 = ols(form,data=df).fit()
print(modelo1.summary())

                            OLS Regression Results                            
Dep. Variable:          MarketCap_log   R-squared:                       0.497
Model:                            OLS   Adj. R-squared:                  0.469
Method:                 Least Squares   F-statistic:                     18.06
Date:                Sun, 08 Mar 2026   Prob (F-statistic):           2.65e-55
Time:                        11:55:03   Log-Likelihood:                -600.59
No. Observations:                 503   AIC:                             1255.
Df Residuals:                     476   BIC:                             1369.
Df Model:                          26                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

In [28]:
form2 = ols_formula(df, "MarketCap_log", 'Ticker', 'random1', 'random2', 'debtToEquity_log', 'currentRatio', 'trailingPegRatio_log', 'ReturnOnAssets', 'operatingMargins', 'revenueGrowth')
modelo2 = ols(form2,data=df).fit()
print(modelo2.summary())

                            OLS Regression Results                            
Dep. Variable:          MarketCap_log   R-squared:                       0.491
Model:                            OLS   Adj. R-squared:                  0.473
Method:                 Least Squares   F-statistic:                     25.99
Date:                Sun, 08 Mar 2026   Prob (F-statistic):           1.00e-59
Time:                        12:07:02   Log-Likelihood:                -603.09
No. Observations:                 503   AIC:                             1244.
Df Residuals:                     484   BIC:                             1324.
Df Model:                          18                                         
Covariance Type:            nonrobust                                         
                                       coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

In [6]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# definir matrices de entrada y objetivo
X = df.drop(['MarketCap_log','Ticker'], axis=1)
y = df['MarketCap_log']

# identificar columnas numéricas y categóricas
numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(include='object').columns.tolist()

# preprocesador: escala numéricas y codifica categóricas
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

pipe = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])

pipe.fit(X_train, y_train)
print("R² test:", pipe.score(X_test, y_test))

# comparar predicciones con el conjunto test
from sklearn.metrics import mean_squared_error, r2_score

y_pred = pipe.predict(X_test)
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

import plotly.express as px
fig = px.scatter(x=y_test, y=y_pred,
                 labels={'x':'Real','y':'Predicho'},
                 title='Pred vs Real en test')
fig.add_shape(type='line', x0=y_test.min(), y0=y_test.min(),
             x1=y_test.max(), y1=y_test.max(),
             line=dict(color='red', dash='dash'))
fig.show()

R² test: 0.631620788617334
MSE: 0.5801693433404159
R2: 0.631620788617334


In [7]:
# Cluster: sobre/sub predicción respecto a la identidad
residuos = y_pred - y_test
cluster = pd.Series(['Sobrepredicción' if r > 0 else 'Subpredicción' for r in residuos], 
                     index=y_test.index)

# Visualizar con los 2 clusters
fig = px.scatter(x=y_test, y=y_pred, color=cluster,
                 labels={'x':'Real','y':'Predicho','color':'Cluster'},
                 title='Predicciones vs Reales (Clusters)')
fig.add_shape(type='line', x0=y_test.min(), y0=y_test.min(),
             x1=y_test.max(), y1=y_test.max(),
             line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster
print("\nEstadísticas por cluster:")
print(f"Sobrepredicción: {(cluster == 'Sobrepredicción').sum()} casos, residuo medio: {residuos[cluster == 'Sobrepredicción'].mean():.4f}")
print(f"Subpredicción: {(cluster == 'Subpredicción').sum()} casos, residuo medio: {residuos[cluster == 'Subpredicción'].mean():.4f}")


Estadísticas por cluster:
Sobrepredicción: 68 casos, residuo medio: 0.5253
Subpredicción: 58 casos, residuo medio: -0.6739


In [8]:
# Guardar cluster en el dataframe
df['cluster_prediccion'] = None
df.loc[y_test.index, 'cluster_prediccion'] = cluster.values

print("\nDataframe con cluster agregado:")
print(df[['Ticker', 'MarketCap_log', 'cluster_prediccion']].head(10))


Dataframe con cluster agregado:
  Ticker  MarketCap_log cluster_prediccion
0    MMM       4.391990    Sobrepredicción
1    AOS       2.284418               None
2    ABT       5.249025      Subpredicción
3   ABBV       6.008502               None
4    ACN       4.892826               None
5   ADBE       4.776796               None
6    AMD       5.748565               None
7    AES       2.310207               None
8    AFL       4.051207               None
9      A       3.482665    Sobrepredicción


In [9]:
# Reajustar modelo con todos los datos
print("Reajustando modelo con el dataset completo...")

X_completo = df.drop(['MarketCap_log','Ticker','cluster_prediccion'], axis=1)
y_completo = df['MarketCap_log']

pipe_final = Pipeline([
    ('pre', preprocessor),
    ('model', RandomForestRegressor(n_estimators=200, random_state=42))
])

pipe_final.fit(X_completo, y_completo)
r2_final = pipe_final.score(X_completo, y_completo)
print(f"R² sobre dataset completo: {r2_final:.4f}")

Reajustando modelo con el dataset completo...
R² sobre dataset completo: 0.9485


In [10]:
# Recrear clusters con predicciones del modelo final
y_pred_completo = pipe_final.predict(X_completo)
residuos_completo = y_pred_completo - y_completo
cluster_completo = pd.Series(['Sobrepredicción' if r > 0 else 'Subpredicción' for r in residuos_completo], 
                              index=y_completo.index)

# Actualizar columna de cluster
df['cluster_prediccion'] = cluster_completo.values

# Visualizar con plotly
fig = px.scatter(df, x='MarketCap_log', y=y_pred_completo, color='cluster_prediccion',
                 hover_data=['Ticker','Sector'],
                 labels={'x':'MarketCap (Real)','y':'MarketCap (Predicho)','cluster_prediccion':'Cluster'},
                 title='Clusters Sobrepredicción/Subpredicción (Datos Completos)')
fig.add_shape(type='line', x0=y_completo.min(), y0=y_completo.min(),
             x1=y_completo.max(), y1=y_completo.max(),
             line=dict(color='black', dash='dash', width=2))
fig.show()

# Estadísticas por cluster
print("\nEstadísticas de clusters (datos completos):")
print(f"Sobrepredicción: {(df['cluster_prediccion'] == 'Sobrepredicción').sum()} casos, residuo medio: {residuos_completo[cluster_completo == 'Sobrepredicción'].mean():.4f}")
print(f"Subpredicción: {(df['cluster_prediccion'] == 'Subpredicción').sum()} casos, residuo medio: {residuos_completo[cluster_completo == 'Subpredicción'].mean():.4f}")


Estadísticas de clusters (datos completos):
Sobrepredicción: 275 casos, residuo medio: 0.1885
Subpredicción: 228 casos, residuo medio: -0.2190


In [11]:
print("\nDataframe con cluster agregado:")
print(df[['Ticker', 'MarketCap_log', 'cluster_prediccion']].head(10))


Dataframe con cluster agregado:
  Ticker  MarketCap_log cluster_prediccion
0    MMM       4.391990    Sobrepredicción
1    AOS       2.284418    Sobrepredicción
2    ABT       5.249025      Subpredicción
3   ABBV       6.008502      Subpredicción
4    ACN       4.892826      Subpredicción
5   ADBE       4.776796      Subpredicción
6    AMD       5.748565      Subpredicción
7    AES       2.310207    Sobrepredicción
8    AFL       4.051207    Sobrepredicción
9      A       3.482665    Sobrepredicción
